# Project : Sentiment Analysis

In [1]:
import numpy as np
import pandas as pd


import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('datasets/sentiment-analysis.csv')
print(df.columns)
print()
print(df)

Index(['Text, Sentiment, Source, Date/Time, User ID, Location, Confidence Score'], dtype='object')

   Text, Sentiment, Source, Date/Time, User ID, Location, Confidence Score
0   "I love this product!", Positive, Twitter, 202...                     
1   "The service was terrible.", Negative, Yelp Re...                     
2   "This movie is amazing!", Positive, IMDb, 2023...                     
3   "I'm so disappointed with their customer suppo...                     
4   "Just had the best meal of my life!", Positive...                     
..                                                ...                     
93  "I can't stop listening to this song. It's my ...                     
94  "Their website is so confusing and poorly desi...                     
95  "I had an incredible experience at the theme p...                     
96                                                NaN                     
97                                                NaN                     


In [3]:
df[:1]

,"Text, Sentiment, Source, Date/Time, User ID, Location, Confidence Score"
0,"""I love this product!"", Positive, Twitter, 202..."


In [4]:
print(df.isna().sum())

df = df.dropna()
print(df.isna().sum())

Text, Sentiment, Source, Date/Time, User ID, Location, Confidence Score    2
dtype: int64
Text, Sentiment, Source, Date/Time, User ID, Location, Confidence Score    0
dtype: int64


In [5]:
data = df['Text, Sentiment, Source, Date/Time, User ID, Location, Confidence Score'].str.split(',', expand=True)
data.columns = ['text', 'sentiment', 'source', 'date/time', 'ID', 'location', 'confidence_score']
data.head(20)

,text,sentiment,source,date/time,ID,location,confidence_score
0,"""I love this product!""",Positive,Twitter,2023-06-15 09:23:14,@user123,New York,0.85
1,"""The service was terrible.""",Negative,Yelp Reviews,2023-06-15 11:45:32,user456,Los Angeles,0.65
2,"""This movie is amazing!""",Positive,IMDb,2023-06-15 14:10:22,moviefan789,London,0.92
3,"""I'm so disappointed with their customer suppo...",Negative,Online Forum,2023-06-15 17:35:11,forumuser1,Toronto,0.78
4,"""Just had the best meal of my life!""",Positive,TripAdvisor,2023-06-16 08:50:59,foodie22,Paris,0.88
5,"""The quality of this product is subpar.""",Negative,Amazon Reviews,2023-06-16 10:15:27,shopper123,San Francisco,0.72
6,"""I can't stop listening to this song. It's inc...",Positive,Spotify,2023-06-16 13:40:18,musiclover456,Berlin,0.91
7,"""Their website is so user-friendly. Love it!""",Positive,Website Testimonial,2023-06-16 16:05:36,testimonialuser1,Sydney,0.87
8,"""I loved the movie! It was fantastic!""",Positive,IMDb,2023-07-02 09:12:34,user123,New York,0.92
9,"""The customer service was terrible.""",Negative,Yelp Reviews,2023-07-02 10:45:21,user456,Los Angeles,0.65


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 96 entries, 0 to 95
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   text              96 non-null     object
 1   sentiment         96 non-null     object
 2   source            96 non-null     object
 3   date/time         96 non-null     object
 4   ID                96 non-null     object
 5   location          96 non-null     object
 6   confidence_score  96 non-null     object
dtypes: object(7)
memory usage: 6.0+ KB


In [7]:
data['confidence_score'] = data['confidence_score'].astype(float)
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 96 entries, 0 to 95
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   text              96 non-null     object 
 1   sentiment         96 non-null     object 
 2   source            96 non-null     object 
 3   date/time         96 non-null     object 
 4   ID                96 non-null     object 
 5   location          96 non-null     object 
 6   confidence_score  96 non-null     float64
dtypes: float64(1), object(6)
memory usage: 6.0+ KB


In [8]:
print('Duplicate items are : ', data.duplicated(subset=['ID']).sum())   # checking duplicates based on user ID


Duplicate items are :  23


In [9]:
print(data.shape)

data = data.drop_duplicates(subset=['ID'])
print(data.shape)

(96, 7)
(73, 7)


In [10]:
data.describe()

,confidence_score
count,73.000000
mean,0.798356
std,0.134092
min,0.550000
25%,0.670000
50%,0.870000
75%,0.920000
max,0.950000


In [11]:
# importing stopwords
import nltk
from nltk.corpus import stopwords               # removes stopwords
stop_words = stopwords.words('english')

# importing porterstemmer
from nltk.stem.porter import PorterStemmer     # stemming brings a word to its root form
st = PorterStemmer()


# importing cleantext to remove symbols
from cleantext import clean                    # can do both stopword removal, and can also do stemming





In [12]:
# printing all the stop words nltk has
print(len(stop_words))
print(stop_words)

198
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 

In [13]:
print(st.stem('hii this is jeet, playing, running, shouring, screams'))

hii this is jeet, playing, running, shouring, scream


In [14]:
nltk.word_tokenize('hii this is jeet !!!')

['hii', 'this', 'is', 'jeet', '!', '!', '!']

In [15]:
help(nltk.word_tokenize)

Help on function word_tokenize in module nltk.tokenize:

word_tokenize(text, language='english', preserve_line=False)
    Return a tokenized copy of *text*,
    using NLTK's recommended word tokenizer
    (currently an improved :class:`.TreebankWordTokenizer`
    along with :class:`.PunktSentenceTokenizer`
    for the specified language).

    :param text: text to split into words
    :type text: str
    :param language: the model name in the Punkt corpus
    :type language: str
    :param preserve_line: A flag to decide whether to sentence tokenize the text or not.
    :type preserve_line: bool



In [16]:
help(nltk.WordNetLemmatizer)

Help on class WordNetLemmatizer in module nltk.stem.wordnet:

class WordNetLemmatizer(builtins.object)
 |  WordNet Lemmatizer
 |
 |  Provides 3 lemmatizer modes: _morphy(), morphy() and lemmatize().
 |
 |  lemmatize() is a permissive wrapper around _morphy().
 |  It returns the shortest lemma found in WordNet,
 |  or the input string unchanged if nothing is found.
 |
 |  >>> from nltk.stem import WordNetLemmatizer as wnl
 |  >>> print(wnl().lemmatize('us', 'n'))
 |  u
 |
 |  >>> print(wnl().lemmatize('Anythinggoeszxcv'))
 |  Anythinggoeszxcv
 |
 |  Methods defined here:
 |
 |  __repr__(self)
 |      Return repr(self).
 |
 |  lemmatize(self, word: str, pos: str = 'n') -> str
 |      Lemmatize `word` by picking the shortest of the possible lemmas,
 |      using the wordnet corpus reader's built-in _morphy function.
 |      Returns the input word unchanged if it cannot be found in WordNet.
 |
 |      >>> from nltk.stem import WordNetLemmatizer as wnl
 |      >>> print(wnl().lemmatize('dog

In [17]:
txt = 'hii this is Jeet !!!, I always go to gym for better body and physiche, I make a very good living.'
clean(txt, stemming=True, stopwords=True, lowercase=True, punct=True)

'hii jeet alway go gym better bodi physich make good live'

In [18]:
# checking for all the keywords used in clean
help(clean)

Help on function clean in module cleantext.cleantext:

clean(
    text: str,
    clean_all: bool = True,
    extra_spaces: bool = False,
    stemming: bool = False,
    stopwords: bool = False,
    lowercase: bool = False,
    numbers: bool = False,
    punct: bool = False,
    reg: str = '',
    reg_replace: str = '',
    stp_lang: str = 'english'
) -> str
    Given a raw string, return cleaned text

    :param text: Input text to clean
    :param clean_all: Execute all cleaning operations
    :param extra_spaces: Remove extra white spaces
    :param stemming: Stem the words
    :param stopwords: Remove stop words
    :param lowercase: Convert to lowercase
    :param numbers: Remove all digits
    :param punct: Remove all punctuations
    :param reg: Regular expression for removing or replacing
    :param reg_replace: Replace the part with regular expression(reg)
    :param stp_lang: Language for stop words
    :return: Cleaned text



# Cleantext vs. NLTK: Which one to use?

The `cleantext` library is incredibly convenient because it acts as an "all-in-one" wrapper. For basic machine learning projects, setting `stemming=True` and `stopwords=True` in `cleantext` is a great way to save time.

However, Data Scientists transition to a heavier library like **NLTK** when they need **Control, Customization, and Advanced Features**.

### 1. Customizing Stopwords
When you use `stopwords=True` in `cleantext`, it uses a hidden, pre-defined list of words to remove. 
*   **The Problem:** In Sentiment Analysis, words like *"not"*, *"don't"*, or *"didn't"* are usually on standard stopword lists. If `cleantext` removes them, "The movie was not good" becomes "movie good"—completely destroying the meaning.
*   **The NLTK Solution:** In NLTK, the stopwords are loaded as a standard Python list. You can easily modify it before cleaning: `my_stopwords.remove("not")` or `my_stopwords.append("custom_word")`.

### 2. Choice of Stemmer vs. Lemmatization
`cleantext` uses one default method for stemming. 
*   **The NLTK Solution:** NLTK gives you options. You can choose the gentle *SnowballStemmer*, the highly aggressive *LancasterStemmer*, or completely bypass stemming to use **Lemmatization** (which turns "better" into "good" using a real linguistic dictionary). `cleantext` does not support Lemmatization.

### 3. Advanced Linguistic Tasks
Cleaning is only the first step of NLP. If you want to move beyond simple word counts, you need NLTK's advanced toolkit:
*   **Part of Speech (POS) Tagging:** Identifying if a word is a Noun, Verb, or Adjective.
*   **Named Entity Recognition (NER):** Automatically identifying that "Apple" in a sentence refers to the company, not the fruit.

---

### Summary
*   **Use `cleantext`** when you want a fast, easy, "black-box" cleaning pipeline and you don't care about fine-tuning the details.
*   **Use `nltk`** when your model's accuracy depends on keeping specific stopwords, choosing specific grammatical roots, or performing advanced text analysis.


In [22]:
data.head(10)

,text,sentiment,source,date/time,ID,location,confidence_score
0,"""I love this product!""",Positive,Twitter,2023-06-15 09:23:14,@user123,New York,0.85
1,"""The service was terrible.""",Negative,Yelp Reviews,2023-06-15 11:45:32,user456,Los Angeles,0.65
2,"""This movie is amazing!""",Positive,IMDb,2023-06-15 14:10:22,moviefan789,London,0.92
3,"""I'm so disappointed with their customer suppo...",Negative,Online Forum,2023-06-15 17:35:11,forumuser1,Toronto,0.78
4,"""Just had the best meal of my life!""",Positive,TripAdvisor,2023-06-16 08:50:59,foodie22,Paris,0.88
5,"""The quality of this product is subpar.""",Negative,Amazon Reviews,2023-06-16 10:15:27,shopper123,San Francisco,0.72
6,"""I can't stop listening to this song. It's inc...",Positive,Spotify,2023-06-16 13:40:18,musiclover456,Berlin,0.91
7,"""Their website is so user-friendly. Love it!""",Positive,Website Testimonial,2023-06-16 16:05:36,testimonialuser1,Sydney,0.87
8,"""I loved the movie! It was fantastic!""",Positive,IMDb,2023-07-02 09:12:34,user123,New York,0.92
10,"""This book made me feel inspired. Highly recom...",Positive,Goodreads,2023-07-02 12:34:56,bookworm789,London,0.88


In [24]:
stop_words = stopwords.words('english')


In [25]:
def transform(text):
    # using clean to covert into lowercase and remove the punctuations
    text = clean(
        text,
        stemming=False,
        lowercase=True,
        punct=True,
        stopwords=False
    )

    # using word_tokenizer
    text = nltk.word_tokenize(text)

    temp = []
    # stemming and removing stopwords
    for word in text:
        if word not in stop_words:
            word = st.stem(word)
            temp.append(word)

    text = " ".join(temp)
    return text

In [27]:
transform('Hello this is Ashura !!!')

'hello ashura'

In [35]:
a = data['text'].apply(transform)
a[2]

'movi amaz'